# Bài 2 — Tìm kiếm: BM25, vector và hybrid

**Tuần 20–21** · Bài tập kèm theo: `ex02_tim_kiem.py`

Bài 1 đã cắt tài liệu thành chunk. Giờ là câu hỏi trung tâm của RAG:

> **Với một câu hỏi, lấy đúng 3 đoạn nào?**

Ba cách, và bài này sẽ **đo** chứ không tranh luận:

| | Tìm theo từ khoá (BM25) | Tìm theo ngữ nghĩa (vector) |
|---|---|---|
| Khớp cái gì | Từ, chính xác từng ký tự | Ý, kể cả khác từ |
| Mạnh ở | Mã lỗi, tên riêng, con số | Cách diễn đạt khác |

Không cần API key.

In [ ]:
import sys
from pathlib import Path

import numpy as np

# Tim thu muc goc du an du notebook duoc mo tu dau
GOC = Path.cwd()
while not (GOC / "pyproject.toml").exists() and GOC != GOC.parent:
    GOC = GOC.parent
P08 = GOC / "curriculum" / "08-rag"
sys.path.insert(0, str(P08))

from tien_ich import bat_utf8, nap_tai_lieu, nap_cau_hoi_vang
from nhung_gia_lap import nhung, nhung_nhieu, giai_thich, SO_CHIEU, TEN_CHIEU

bat_utf8()   # Windows: console cp1252 se lam sap moi lenh print tieng Viet

DOCS = nap_tai_lieu()
VANG = nap_cau_hoi_vang()

print(f"{len(DOCS)} tài liệu · {sum(len(v) for v in DOCS.values()):,} ký tự")
print(f"{len(VANG)} câu hỏi vàng · embedding giả lập {SO_CHIEU} chiều")

In [ ]:
def cat_co_dinh(van_ban, kich_thuoc, chong_lap=0):
    """Giong ex01_chunking.cat_co_dinh - dat o day de notebook chay doc lap."""
    if chong_lap >= kich_thuoc:
        raise ValueError("chong_lap phai nho hon kich_thuoc")
    buoc = kich_thuoc - chong_lap
    return [van_ban[i:i + kich_thuoc] for i in range(0, len(van_ban), buoc)]


def cat_kho(docs, kich_thuoc=600, chong_lap=120):
    """Cat toan bo kho -> (danh sach chunk, danh sach ten file tuong ung)."""
    chunks, nguon = [], []
    for ten, noi_dung in docs.items():
        for c in cat_co_dinh(noi_dung, kich_thuoc, chong_lap):
            chunks.append(c)
            nguon.append(ten)
    return chunks, nguon


def gop(s):
    """Gop moi khoang trang/xuong dong thanh mot dau cach."""
    return " ".join(s.split())


def co_trich_dan(cac_chunk, trich_dan):
    """Doan trich dan co nam trong cac chunk lay ve khong?"""
    kho = gop(" || ".join(cac_chunk))
    return gop(trich_dan) in kho

In [ ]:
# Cắt kho một lần, dùng cho cả notebook
CHUNKS, NGUON = cat_kho(DOCS, 600, 120)
print(f"{len(CHUNKS)} chunk từ {len(DOCS)} tài liệu")

## 1. BM25 — tìm theo từ khoá

Công thức nghe đáng sợ nhưng chỉ có ba ý:

```
diem = Σ  IDF(t) × ( tf × (k1+1) ) / ( tf + k1 × (1 − b + b × L/L_tb) )
        t∈truy vấn
```

| Thành phần | Ý nghĩa |
|---|---|
| `tf` với `k1` | Tần suất từ, **có bão hoà** — xuất hiện 20 lần không đáng gấp 20 lần |
| `IDF` | Từ **hiếm** đáng giá hơn từ phổ biến |
| `L/L_tb` với `b` | **Phạt tài liệu dài** — dài thì dễ chứa từ một cách tình cờ |

`IDF` là lý do BM25 tự động coi nhẹ những từ như "của", "và" mà **không cần
danh sách stopword** nào cả.

In [ ]:
import math
import re
from collections import Counter

MAU_TU = re.compile(r"[0-9a-zà-ỹđ]+(?:[-.][0-9a-zà-ỹđ]+)*")


def tach_tu(s):
    """Ha thuong TRUOC khi khop: dai a-y khong chua chu HOA co dau."""
    return MAU_TU.findall(s.lower())


class BM25:
    def __init__(self, tai_lieu, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.cac_tu = [tach_tu(t) for t in tai_lieu]
        self.tan_suat = [Counter(t) for t in self.cac_tu]
        self.do_dai = [len(t) for t in self.cac_tu]
        self.N = len(tai_lieu)
        self.do_dai_tb = sum(self.do_dai) / self.N if self.N else 0.0
        self.df = Counter()
        for t in self.cac_tu:
            self.df.update(set(t))     # MOI tai lieu dem MOT lan

    def _idf(self, tu):
        df = self.df.get(tu, 0)
        return math.log(1 + (self.N - df + 0.5) / (df + 0.5))

    def diem(self, truy_van):
        tv = tach_tu(truy_van)
        out = []
        for i in range(self.N):
            tf_i = self.tan_suat[i]
            chuan = 1 - self.b + self.b * self.do_dai[i] / (self.do_dai_tb or 1)
            out.append(sum(
                self._idf(t) * tf_i[t] * (self.k1 + 1) / (tf_i[t] + self.k1 * chuan)
                for t in tv if tf_i.get(t)
            ))
        return np.array(out)

    def tim(self, truy_van, k=3):
        d = self.diem(truy_van)
        return sorted(range(len(d)), key=lambda i: (-d[i], i))[:k]


bm = BM25(CHUNKS)
print("Sẵn sàng.", len(bm.df), "từ trong từ điển")

In [ ]:
# Từ nào "đáng giá" nhất và ít giá trị nhất trong kho này?
xep = sorted(bm.df.items(), key=lambda kv: kv[1])
print("Hiếm nhất (IDF cao):")
for tu, df in xep[:5]:
    print(f"   {tu:<14} có mặt trong {df}/{bm.N} chunk   IDF={bm._idf(tu):.2f}")
print("\nPhổ biến nhất (IDF thấp):")
for tu, df in xep[-5:]:
    print(f"   {tu:<14} có mặt trong {df}/{bm.N} chunk   IDF={bm._idf(tu):.2f}")

## 2. Thử một câu hỏi mã lỗi

In [ ]:
cau = "Mã lỗi VX-204 nghĩa là gì?"

print("BM25:")
for hang, i in enumerate(bm.tim(cau, 3), 1):
    print(f"   {hang}. [{NGUON[i]}] {gop(CHUNKS[i])[:80]}...")

M = nhung_nhieu(CHUNKS)
top_vec = np.argsort(-(M @ nhung(cau)))[:3]
print("\nEmbedding:")
for hang, i in enumerate(top_vec, 1):
    print(f"   {hang}. [{NGUON[i]}] {gop(CHUNKS[i])[:80]}...")

BM25 nhảy đúng vào đoạn chứa `VX-204`. Embedding thì lang thang.

Lý do đã thấy ở bài 1: với embedding, `VX-204` gần như không có tín hiệu —
model chưa từng thấy chuỗi này đủ nhiều để học ý nghĩa của nó.

> ⚠️ Đây **không phải** giới hạn riêng của bản giả lập. Embedding thật cũng yếu
> với mã đơn hàng, số hợp đồng, SKU, tên biến, mã lỗi — tức là **chính xác
> những thứ người dùng doanh nghiệp hay hỏi nhất**.

## 3. Bây giờ thử câu diễn đạt khác

In [ ]:
cau2 = "Tôi bị bệnh không đến chỗ làm được thì có mất lương không?"
DUNG = "tối đa 3 ngày mỗi năm mà không cần giấy tờ"

for ten, idx in [("BM25", bm.tim(cau2, 3)),
                 ("Embedding", np.argsort(-(M @ nhung(cau2)))[:3])]:
    lay = [CHUNKS[i] for i in idx]
    dat = "✅ LẤY ĐƯỢC" if co_trich_dan(lay, DUNG) else "❌ TRƯỢT"
    print(f"{ten:<12} {dat}")
    for hang, i in enumerate(idx, 1):
        print(f"   {hang}. [{NGUON[i]}] {gop(CHUNKS[i])[:70]}...")
    print()

Đảo chiều hoàn toàn. Câu hỏi không có một từ nào là "nghỉ ốm", nên BM25 mù;
embedding vẫn bắt được vì cùng khái niệm *ốm đau*.

**Hai phương pháp hỏng ở hai chỗ hoàn toàn khác nhau.** Đó là lý do tồn tại của
hybrid search — không phải vì "gộp lại thì hay hơn".

## 4. Đo trên cả bộ câu hỏi vàng

Đủ ví dụ lẻ. Giờ đo có hệ thống, tách theo **loại câu hỏi**.

In [ ]:
def recall_mot_cau(cau, idx):
    lay = [CHUNKS[i] for i in idx]
    return sum(1 for td in cau["trich_dan"] if co_trich_dan(lay, td)) / len(cau["trich_dan"])


def do_toan_bo(ham_tim, k=3):
    """ham_tim(cau_hoi, k) -> danh sách chỉ số chunk."""
    theo_loai, tat_ca = {}, []
    for c in VANG:
        if not c["trich_dan"]:
            continue
        r = recall_mot_cau(c, ham_tim(c["cau_hoi"], k))
        theo_loai.setdefault(c["loai"], []).append(r)
        tat_ca.append(r)
    return sum(tat_ca) / len(tat_ca), {l: sum(v) / len(v) for l, v in theo_loai.items()}


def tim_bm25(cau, k=3):
    return bm.tim(cau, k)


def tim_vector(cau, k=3):
    return list(np.argsort(-(M @ nhung(cau)))[:k])


LOAI = ["de", "ma_loi", "dien_dat_khac", "nhieu_doan", "kho"]

tong_b, loai_b = do_toan_bo(tim_bm25)
tong_v, loai_v = do_toan_bo(tim_vector)

print(f'{"loại câu hỏi":<18}{"BM25":>8}{"Vector":>8}   ai thắng')
for l in LOAI:
    b, v = loai_b[l], loai_v[l]
    ai = "BM25" if b > v else ("Vector" if v > b else "hoà")
    print(f"{l:<18}{b:>8.2f}{v:>8.2f}   {ai}")
print(f'{"TRUNG BÌNH":<18}{tong_b:>8.2f}{tong_v:>8.2f}')

Bảng này là **toàn bộ luận điểm của bài học**:

- `ma_loi`: BM25 **1.00**, vector **0.00**
- `dien_dat_khac`: vector **1.00**, BM25 thấp hơn

Trung bình BM25 nhỉnh hơn — nhưng chọn BM25 nghĩa là **vứt bỏ hoàn toàn** nhóm
câu diễn đạt khác. Mà trong thực tế, người dùng gõ vào ô tìm kiếm **bằng chữ
của họ**, không phải bằng chữ trong tài liệu.

## 5. Hybrid — Reciprocal Rank Fusion

Không thể cộng thẳng điểm BM25 với điểm cosine: một cái chạy từ 0 đến vô cùng,
cái kia từ −1 đến 1. BM25 sẽ nuốt chửng cosine.

**RRF vứt bỏ điểm gốc, chỉ giữ thứ hạng:**

```
điểm(mục) = Σ  1 / (k0 + thứ_hạng)          thứ hạng bắt đầu từ 1
```

Vì chỉ dùng thứ hạng nên không cần chuẩn hoá gì cả.

In [ ]:
def hop_nhat_rrf(cac_bang, k=3, k0=60):
    diem = {}
    for bang in cac_bang:
        for hang, i in enumerate(bang, start=1):
            diem[i] = diem.get(i, 0.0) + 1.0 / (k0 + hang)
    return sorted(diem, key=lambda i: (-diem[i], i))[:k]


def tim_hybrid(cau, k=3, sau=None):
    # `sau` = lấy bao nhiêu ứng viên từ MỖI bên trước khi hợp nhất.
    # Mặc định lấy TOÀN BỘ bảng xếp hạng - đó là cách RRF nguyên bản.
    sau = sau or len(CHUNKS)
    return hop_nhat_rrf([tim_bm25(cau, sau), tim_vector(cau, sau)], k)


tong_h, loai_h = do_toan_bo(tim_hybrid)

print(f'{"loại câu hỏi":<18}{"BM25":>8}{"Vector":>8}{"Hybrid":>8}')
for l in LOAI:
    print(f"{l:<18}{loai_b[l]:>8.2f}{loai_v[l]:>8.2f}{loai_h[l]:>8.2f}")
print(f'{"TRUNG BÌNH":<18}{tong_b:>8.2f}{tong_v:>8.2f}{tong_h:>8.2f}')

In [ ]:
# RRF nhạy với việc bạn hợp nhất SÂU bao nhiêu
print(f'{"độ sâu hợp nhất":<22}{"tổng":>8}{"ma_loi":>9}{"nhieu_doan":>12}')
for sau in (2, 4, 8, len(CHUNKS)):
    t, d = do_toan_bo(lambda c, k, s=sau: tim_hybrid(c, k, s))
    nhan = "toàn bộ" if sau == len(CHUNKS) else f"top-{sau} mỗi bên"
    print(f'{nhan:<22}{t:>8.2f}{d["ma_loi"]:>9.2f}{d["nhieu_doan"]:>12.2f}')

Hợp nhất càng sâu càng ổn định. `top-2 mỗi bên` thấp hơn các dòng còn lại
— vì gộp hai danh sách 2 phần tử thì gần như không có gì để mà gộp.

> ⚠️ **Nhưng đừng đọc quá nhiều vào chênh lệch nhỏ.** Bộ test này chỉ có 16 câu
> được chấm, nên **một câu lật làm điểm trung bình đổi ~6 điểm phần trăm**.
> Chênh lệch 0.91 với 0.94 ở đây hoàn toàn có thể chỉ là một câu hỏi duy nhất.
>
> Đây là bài học của Phase 04 quay lại: một con số không kèm cỡ mẫu thì không
> nói lên điều gì. Trước khi ăn mừng vì "cải thiện 3 điểm", hãy hỏi **3 điểm đó
> là mấy câu hỏi?** Bộ test thật cho hệ thống của bạn nên có 50–200 câu.

## 6. ⚠️ Hybrid không miễn phí

Nhìn cột `Hybrid` ở dòng `ma_loi` và so với cột `BM25`.

Trung bình tăng, nhưng nhóm mã lỗi **tụt xuống** so với BM25 đơn thuần. RRF coi
hai bảng xếp hạng quan trọng như nhau — nên khi một bên chắc chắn đúng và bên
kia mù tịt, nó vẫn kéo kết quả của bên mù tịt lên, đẩy đoạn đúng ra khỏi top-3.

In [ ]:
cau = "Mã lỗi VX-204 nghĩa là gì?"
DUNG_204 = "Xung đột dữ liệu chấm công"

for ten, idx in [("BM25", tim_bm25(cau, 3)),
                 ("Vector", tim_vector(cau, 3)),
                 ("Hybrid", tim_hybrid(cau, 3))]:
    lay = [CHUNKS[i] for i in idx]
    print(f'{ten:<8} {"✅" if co_trich_dan(lay, DUNG_204) else "❌"}  '
          f"đoạn lấy về: {[NGUON[i][:18] for i in idx]}")

> 📌 Nếu chỉ nhìn con số trung bình, bạn kết luận *"hybrid tốt hơn, xong"* và
> **không hề biết mình vừa làm hỏng cả nhóm câu hỏi mã lỗi** — nhóm mà nhân viên
> hỗ trợ kỹ thuật hỏi nhiều nhất.
>
> Đây chính xác là bài học "trung bình tăng ≠ tốt hơn" của Phase 07, lặp lại ở
> một chỗ khác. Nó sẽ còn lặp lại nữa ở bài 4.

**Cách xử lý trong thực tế:** phát hiện câu hỏi có chứa mã định danh (regex đơn
giản) và cho BM25 trọng số cao hơn với riêng những câu đó. RRF có biến thể
gán trọng số cho từng bảng — nhưng bạn chỉ biết cần làm thế **sau khi đã đo**.

## 7. Rerank có đáng thêm không?

Rerank = lấy rộng bằng cách rẻ, rồi dùng **cross-encoder** đọc từng cặp
(câu hỏi, đoạn) để chấm lại và giữ top-3.

| | Bi-encoder (retrieval) | Cross-encoder (rerank) |
|---|---|---|
| Cách làm | Nhúng câu hỏi và đoạn **riêng biệt** | Đọc **cùng lúc** cả hai |
| Tốc độ | Rất nhanh — một phép nhân ma trận | Chậm — mỗi cặp một lần chạy model |

Vì chậm nên không ai quét cả kho bằng cross-encoder.

**Trước khi thêm rerank, hãy đo xem nó có chỗ để cải thiện không:**

In [ ]:
for k in (3, 5, 10, 20):
    t, _ = do_toan_bo(tim_hybrid, k)
    print(f"recall@{k:<3} = {t:.2f}")

Cách đọc bảng này:

- **recall@20 cao hơn hẳn recall@3** → đoạn đúng *có* được lấy về, chỉ là xếp
  hạng thấp. **Rerank sẽ giúp** — đó đúng là việc nó làm.
- **recall@20 cũng thấp** → đoạn đúng chưa từng được lấy về. Rerank **vô ích**;
  phải quay lại sửa chunking hoặc embedding.

> 📌 Đây là cách quyết định kỹ thuật dựa trên số đo, thay vì thêm một thành phần
> vì "ai cũng bảo nên có rerank". Mỗi thành phần bạn thêm vào đều là thêm độ trễ,
> thêm chi phí và thêm một chỗ có thể hỏng.

## 8. MRR — thứ hạng có quan trọng không?

Recall chỉ hỏi *"có lấy được không"*. MRR hỏi *"nó nằm ở hạng mấy"*.

Với RAG, thứ hạng rất quan trọng vì hai lý do: bạn chỉ nhét được vài đoạn vào
prompt, và model **chú ý đoạn đầu nhiều hơn đoạn giữa** (bài 3 sẽ nói kỹ).

In [ ]:
def mrr_mot_cau(cau, idx):
    """1/hạng của chunk ĐẦU TIÊN chứa một đoạn trích đúng."""
    for hang, i in enumerate(idx, start=1):
        if any(co_trich_dan([CHUNKS[i]], td) for td in cau["trich_dan"]):
            return 1.0 / hang
    return 0.0


print(f'{"cách tìm":<12}{"recall@10":>11}{"MRR@10":>9}')
for ten, f in [("BM25", tim_bm25), ("Vector", tim_vector), ("Hybrid", tim_hybrid)]:
    rs, ms = [], []
    for c in VANG:
        if not c["trich_dan"]:
            continue
        idx = f(c["cau_hoi"], 10)
        rs.append(recall_mot_cau(c, idx))
        ms.append(mrr_mot_cau(c, idx))
    print(f"{ten:<12}{sum(rs)/len(rs):>11.2f}{sum(ms)/len(ms):>9.2f}")

## Tóm tắt

| Điều | Nhớ gì |
|---|---|
| **BM25** | Vô địch với mã lỗi, tên riêng, con số. Chưa chết, và sẽ không chết |
| **Embedding** | Vô địch khi người dùng diễn đạt khác tài liệu |
| **RRF** | Cộng nghịch đảo **thứ hạng**, nên không cần chuẩn hoá điểm |
| **Hybrid có giá** | Nó kéo lùi nhóm câu mà một phương pháp vốn làm hoàn hảo |
| **Rerank** | Chỉ giúp khi recall@20 ≫ recall@3. Đo trước khi thêm |
| **MRR** | Recall hỏi "có không", MRR hỏi "hạng mấy" |

**Bài tập:** `curriculum/08-rag/exercises/ex02_tim_kiem.py`

```powershell
pytest tests/phase08/test_ex02.py -v
```

➡️ Bài tiếp: **[03 — Sinh câu trả lời có trích dẫn](03_sinh_cau_tra_loi_citation.ipynb)**